In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip
/kaggle/input/competitions/word2vec-nlp-tutorial/sampleSubmission.csv
/kaggle/input/competitions/word2vec-nlp-tutorial/unlabeledTrainData.tsv.zip
/kaggle/input/competitions/word2vec-nlp-tutorial/labeledTrainData.tsv.zip


In [2]:
import logging
import os
import sys
import time
import math
import re

import pandas as pd
import torch
from torch import nn
from torch import optim
from torch.nn import functional as F
from torch.nn.utils.rnn import pad_sequence
from tqdm import tqdm
from bs4 import BeautifulSoup
from collections import defaultdict
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

# ===================== Kaggle路径配置 =====================
TRAIN_PATH = "/kaggle/input/competitions/word2vec-nlp-tutorial/labeledTrainData.tsv.zip"
TEST_PATH = "/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip"
OUTPUT_DIR = "/kaggle/working/result"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ===================== 超参 =====================
num_epochs = 10
MAX_SEQ_LEN = 512    # 句子最大截断长度，和位置编码max_len对齐
embed_size = 128
num_hiddens = 128
num_layers = 2
num_head = 4
dim_feedforward = 512
batch_size = 32
labels = 2
lr = 1e-4
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')


def review_to_wordlist(review, remove_stopwords=False):
    review_text = BeautifulSoup(review, "lxml").get_text()
    review_text = re.sub("[^a-zA-Z]", " ", review_text)
    words = review_text.lower().split()
    return words  # 修复：返回单词list，不再返回字符串！！！


class Vocab:
    def __init__(self, tokens=None):
        self.idx_to_token = list()
        self.token_to_idx = dict()

        if tokens is not None:
            if "<unk>" not in tokens:
                tokens = tokens + ["<unk>"]
            for token in tokens:
                self.idx_to_token.append(token)
                self.token_to_idx[token] = len(self.idx_to_token) - 1
            self.unk = self.token_to_idx['<unk>']

    @classmethod
    def build(cls, train_sents, test_sents, min_freq=1, reserved_tokens=None):
        token_freqs = defaultdict(int)
        for sentence in train_sents:
            for token in sentence:
                token_freqs[token] += 1
        for sentence in test_sents:
            for token in sentence:
                token_freqs[token] += 1

        uniq_tokens = ["<unk>"] + (reserved_tokens if reserved_tokens else [])
        uniq_tokens += [token for token, freq in token_freqs.items()
                        if freq >= min_freq and token != "<unk>"]
        return cls(uniq_tokens)

    def __len__(self):
        return len(self.idx_to_token)

    def __getitem__(self, token):
        return self.token_to_idx.get(token, self.unk)

    def convert_tokens_to_ids(self, tokens):
        return [self[token] for token in tokens]


def length_to_mask(lengths, device):
    max_length = torch.max(lengths)
    mask = torch.arange(max_length, device=device).expand(lengths.shape[0], max_length) < lengths.unsqueeze(1)
    return mask


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=512):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)
        self.register_buffer('pe', pe)
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, x):
        seq_len = x.size(0)
        x = x + self.pe[:seq_len, :]
        return self.dropout(x)


class Transformer(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_class,
                 dim_feedforward=512, num_head=4, num_layers=2, dropout=0.1, max_len=512, activation: str = "relu"):
        super(Transformer, self).__init__()
        assert embedding_dim == hidden_dim, "embedding_dim必须等于hidden_dim(d_model)"
        assert hidden_dim % num_head == 0, "hidden_dim必须可以被num_head整除"

        self.embedding_dim = embedding_dim
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.position_embedding = PositionalEncoding(embedding_dim, dropout, max_len)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_head,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation=activation,
            batch_first=False
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers)
        self.output = nn.Linear(hidden_dim, num_class)

    def forward(self, inputs, lengths):
        # inputs: [batch, seq_len] batch_first=True
        inputs = torch.transpose(inputs, 0, 1)  # [seq_len, batch]
        hidden_states = self.embeddings(inputs)
        hidden_states = self.position_embedding(hidden_states)

        attention_mask = length_to_mask(lengths, inputs.device) == False
        hidden_states = self.transformer(hidden_states, src_key_padding_mask=attention_mask)
        hidden_states = hidden_states[0, :, :]
        output = self.output(hidden_states)
        log_probs = F.log_softmax(output, dim=1)
        return log_probs


class TransformerDataset(torch.utils.data.Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, i):
        return self.data[i]


# 训练/验证集collate_fn
def collate_fn(examples):
    lengths = torch.tensor([len(ex[0]) for ex in examples])
    inputs = [torch.tensor(ex[0]) for ex in examples]
    targets = torch.tensor([ex[1] for ex in examples], dtype=torch.long)
    inputs = pad_sequence(inputs, batch_first=True)
    return inputs, lengths, targets


# 测试集collate_fn
def collate_fn_test(examples):
    lengths = torch.tensor([len(ex) for ex in examples])
    inputs = [torch.tensor(ex) for ex in examples]
    inputs = pad_sequence(inputs, batch_first=True)
    return inputs, lengths


if __name__ == '__main__':
    program = os.path.basename(sys.argv[0])
    logger = logging.getLogger(program)
    logging.basicConfig(format='%(asctime)s: %(levelname)s: %(message)s')
    logging.root.setLevel(level=logging.INFO)

    train = pd.read_csv(TRAIN_PATH, header=0, delimiter="\t", quoting=3)
    test = pd.read_csv(TEST_PATH, header=0, delimiter="\t", quoting=3)

    clean_train_tokens, train_labels = [], []
    for i, review in enumerate(train["review"]):
        tokens = review_to_wordlist(review)
        tokens = tokens[:MAX_SEQ_LEN]  # 长句子截断！
        clean_train_tokens.append(tokens)
        train_labels.append(train["sentiment"][i])

    clean_test_tokens = []
    for review in test["review"]:
        tokens = review_to_wordlist(review)
        tokens = tokens[:MAX_SEQ_LEN]
        clean_test_tokens.append(tokens)

    vocab = Vocab.build(clean_train_tokens, clean_test_tokens)
    logger.info(f"vocab size: {len(vocab)}")

    train_reviews = [(vocab.convert_tokens_to_ids(sentence), train_labels[i])
                     for i, sentence in enumerate(clean_train_tokens)]
    test_reviews = [vocab.convert_tokens_to_ids(sentence)
                    for sentence in clean_test_tokens]

    train_reviews, val_reviews, train_labels, val_labels = train_test_split(
        train_reviews, train_labels, test_size=0.2, random_state=0)

    net = Transformer(
        vocab_size=len(vocab),
        embedding_dim=embed_size,
        hidden_dim=num_hiddens,
        num_class=labels,
        dim_feedforward=dim_feedforward,
        num_head=num_head,
        num_layers=num_layers,
        dropout=0.1,
        max_len=512,
        activation="relu"
    )
    net.to(device)

    loss_function = nn.CrossEntropyLoss()
    optimizer = optim.Adam(net.parameters(), lr=lr)

    train_set = TransformerDataset(train_reviews)
    val_set = TransformerDataset(val_reviews)
    test_set = TransformerDataset(test_reviews)

    train_iter = torch.utils.data.DataLoader(train_set, collate_fn=collate_fn, batch_size=batch_size, shuffle=True)
    val_iter = torch.utils.data.DataLoader(val_set, collate_fn=collate_fn, batch_size=batch_size, shuffle=False)
    test_iter = torch.utils.data.DataLoader(test_set, collate_fn=collate_fn_test, batch_size=batch_size, shuffle=False)

    for epoch in range(num_epochs):
        start = time.time()
        train_loss, val_losses = 0.0, 0.0
        train_acc, val_acc = 0.0, 0.0
        n, m = 0, 0

        net.train()
        with tqdm(total=len(train_iter), desc=f'Epoch {epoch}') as pbar:
            for feature, lengths, label in train_iter:
                n += 1
                net.zero_grad()
                feature = feature.to(device)
                lengths = lengths.to(device)
                label = label.to(device)
                score = net(feature, lengths)
                loss = loss_function(score, label)
                loss.backward()
                optimizer.step()

                train_acc += accuracy_score(torch.argmax(score.cpu().data, dim=1), label.cpu())
                train_loss += loss.item()

                pbar.set_postfix({
                    'train_loss': f'{train_loss / n:.4f}',
                    'train_acc': f'{train_acc / n:.2f}'
                })
                pbar.update(1)

        net.eval()
        with torch.no_grad():
            for val_feature, val_length, val_label in val_iter:
                m += 1
                val_feature = val_feature.to(device)
                val_length = val_length.to(device)
                val_label = val_label.to(device)
                val_score = net(val_feature, val_length)
                val_loss = loss_function(val_score, val_label)
                val_acc += accuracy_score(torch.argmax(val_score.cpu().data, dim=1), val_label.cpu())
                val_losses += val_loss.item()

        end = time.time()
        runtime = end - start
        print(f"Epoch {epoch} | train_loss:{train_loss / n:.4f} train_acc:{train_acc / n:.4f} | val_loss:{val_losses / m:.4f} val_acc:{val_acc / m:.4f} | time:{runtime:.2f}s")

    # 预测测试集
    net.eval()
    test_pred = []
    with torch.no_grad():
        with tqdm(total=len(test_iter), desc='Prediction') as pbar:
            for test_feature, test_len in test_iter:
                test_feature = test_feature.to(device)
                test_len = test_len.to(device)
                test_score = net(test_feature, test_len)
                test_pred.extend(torch.argmax(test_score.cpu().data, dim=1).numpy().tolist())
                pbar.update(1)

    result_output = pd.DataFrame(data={"id": test["id"], "sentiment": test_pred})
    save_path = os.path.join(OUTPUT_DIR, "transformer.csv")
    result_output.to_csv(save_path, index=False, quoting=3)
    logger.info(f'result saved to {save_path}')


2026-08-20 07:29:13,593: INFO: vocab size: 98827
/tmp/ipykernel_58/3904982183.py:128: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers)
Epoch 0: 100%|██████████| 625/625 [00:33<00:00, 18.68it/s, train_loss=0.6858, train_acc=0.56]


Epoch 0 | train_loss:0.6858 train_acc:0.5582 | val_loss:0.6163 val_acc:0.6626 | time:35.50s


Epoch 1: 100%|██████████| 625/625 [00:32<00:00, 19.07it/s, train_loss=0.5547, train_acc=0.71]


Epoch 1 | train_loss:0.5547 train_acc:0.7116 | val_loss:0.5040 val_acc:0.7566 | time:34.85s


Epoch 2: 100%|██████████| 625/625 [00:33<00:00, 18.46it/s, train_loss=0.4661, train_acc=0.78]


...

Epoch 8: 100%|██████████| 625/625 [00:34<00:00, 18.14it/s, train_loss=0.3002, train_acc=0.88]


Epoch 8 | train_loss:0.3002 train_acc:0.8753 | val_loss:0.3696 val_acc:0.8497 | time:36.61s


Epoch 9: 100%|██████████| 625/625 [00:34<00:00, 18.10it/s, train_loss=0.2863, train_acc=0.88]


Epoch 9 | train_loss:0.2863 train_acc:0.8810 | val_loss:0.3605 val_acc:0.8531 | time:36.71s


Prediction: 100%|██████████| 782/782 [00:10<00:00, 74.11it/s]
2026-08-20 07:35:34,927: INFO: result saved to /kaggle/working/result/transformer.csv


In [4]:
import logging
import os
import sys
import time
import math
import re

import pandas as pd
import torch
from torch import nn
from torch import optim
from torch.nn import functional as F
from torch.nn.utils.rnn import pad_sequence
from tqdm import tqdm
from bs4 import BeautifulSoup
from collections import defaultdict
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

# ===================== Kaggle路径配置 =====================
TRAIN_PATH = "/kaggle/input/competitions/word2vec-nlp-tutorial/labeledTrainData.tsv.zip"
TEST_PATH = "/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip"
OUTPUT_DIR = "/kaggle/working/result"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ===================== 优化后超参 =====================
num_epochs = 12
MAX_SEQ_LEN = 512
embed_size = 128
num_hiddens = 128
num_layers = 3
num_head = 4
dim_feedforward = 1024
batch_size = 32
labels = 2
lr = 2e-4
warmup_steps = 400
weight_decay = 1e-5
grad_clip = 1.0
min_word_freq = 5
dropout_rate = 0.2
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

stopwords = {"a","an","the","and","or","but","is","are","was","were","be","been","in","on","at","to","of","for","with","by"}

def review_to_wordlist(review, remove_stopwords=True):
    review_text = BeautifulSoup(review, "lxml").get_text()
    review_text = re.sub("[^a-zA-Z]", " ", review_text)
    words = review_text.lower().split()
    if remove_stopwords:
        words = [w for w in words if w not in stopwords]
    return words


class Vocab:
    def __init__(self, tokens=None):
        self.idx_to_token = list()
        self.token_to_idx = dict()

        if tokens is not None:
            if "<unk>" not in tokens:
                tokens = tokens + ["<unk>"]
            for token in tokens:
                self.idx_to_token.append(token)
                self.token_to_idx[token] = len(self.idx_to_token) - 1
            self.unk = self.token_to_idx['<unk>']

    @classmethod
    def build(cls, train_sents, test_sents, min_freq=1, reserved_tokens=None):
        token_freqs = defaultdict(int)
        for sentence in train_sents:
            for token in sentence:
                token_freqs[token] += 1
        for sentence in test_sents:
            for token in sentence:
                token_freqs[token] += 1

        uniq_tokens = ["<unk>"] + (reserved_tokens if reserved_tokens else [])
        uniq_tokens += [token for token, freq in token_freqs.items()
                        if freq >= min_freq and token != "<unk>"]
        return cls(uniq_tokens)

    def __len__(self):
        return len(self.idx_to_token)

    def __getitem__(self, token):
        return self.token_to_idx.get(token, self.unk)

    def convert_tokens_to_ids(self, tokens):
        return [self[token] for token in tokens]


def length_to_mask(lengths, device):
    max_length = torch.max(lengths)
    mask = torch.arange(max_length, device=device).expand(lengths.shape[0], max_length) < lengths.unsqueeze(1)
    return mask


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=512):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)
        self.register_buffer('pe', pe)
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, x):
        seq_len = x.size(0)
        x = x + self.pe[:seq_len, :]
        return self.dropout(x)


class Transformer(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_class,
                 dim_feedforward=512, num_head=4, num_layers=2, dropout=0.1, max_len=512, activation: str = "relu"):
        super(Transformer, self).__init__()
        assert embedding_dim == hidden_dim, "embedding_dim必须等于hidden_dim(d_model)"
        assert hidden_dim % num_head == 0, "hidden_dim必须可以被num_head整除"

        self.d_model = embedding_dim
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.position_embedding = PositionalEncoding(embedding_dim, dropout, max_len)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_head,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation=activation,
            batch_first=False
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers)
        self.output = nn.Linear(hidden_dim, num_class)

    def forward(self, inputs, lengths):
        # inputs: [batch, seq_len]
        batch_size = inputs.size(0)
        inputs = torch.transpose(inputs, 0, 1)  # [seq_len, batch]
        hidden_states = self.embeddings(inputs)
        # embedding缩放，原始transformer论文
        hidden_states = hidden_states * math.sqrt(self.d_model)
        hidden_states = self.position_embedding(hidden_states)

        attention_mask = length_to_mask(lengths, inputs.device) == False
        hidden_states = self.transformer(hidden_states, src_key_padding_mask=attention_mask)
        hidden_states = hidden_states.transpose(0,1) # [batch, seq_len, dim]

        # ========= 关键优化：均值池化替代取第0个token =========
        mask = length_to_mask(lengths, inputs.device).unsqueeze(-1)
        sum_hidden = torch.sum(hidden_states * mask, dim=1)
        mean_hidden = sum_hidden / lengths.unsqueeze(-1).float()

        output = self.output(mean_hidden)
        # CrossEntropyLoss内部自带log‑softmax，此处直接返回logits！
        return output


class TransformerDataset(torch.utils.data.Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, i):
        return self.data[i]


def collate_fn(examples):
    lengths = torch.tensor([len(ex[0]) for ex in examples])
    inputs = [torch.tensor(ex[0]) for ex in examples]
    targets = torch.tensor([ex[1] for ex in examples], dtype=torch.long)
    inputs = pad_sequence(inputs, batch_first=True)
    return inputs, lengths, targets


def collate_fn_test(examples):
    lengths = torch.tensor([len(ex) for ex in examples])
    inputs = [torch.tensor(ex) for ex in examples]
    inputs = pad_sequence(inputs, batch_first=True)
    return inputs, lengths


# 学习率预热+衰减调度
def get_lr(step, warmup, d_model):
    return d_model ** (-0.5) * min(step ** (-0.5), step * warmup ** (-1.5))


if __name__ == '__main__':
    program = os.path.basename(sys.argv[0])
    logger = logging.getLogger(program)
    logging.basicConfig(format='%(asctime)s: %(levelname)s: %(message)s')
    logging.root.setLevel(level=logging.INFO)

    train = pd.read_csv(TRAIN_PATH, header=0, delimiter="\t", quoting=3)
    test = pd.read_csv(TEST_PATH, header=0, delimiter="\t", quoting=3)

    clean_train_tokens, train_labels = [], []
    for i, review in enumerate(train["review"]):
        tokens = review_to_wordlist(review)
        tokens = tokens[:MAX_SEQ_LEN]
        clean_train_tokens.append(tokens)
        train_labels.append(train["sentiment"][i])

    clean_test_tokens = []
    for review in test["review"]:
        tokens = review_to_wordlist(review)
        tokens = tokens[:MAX_SEQ_LEN]
        clean_test_tokens.append(tokens)

    vocab = Vocab.build(clean_train_tokens, clean_test_tokens, min_freq=min_word_freq)
    logger.info(f"vocab size: {len(vocab)}")

    train_reviews = [(vocab.convert_tokens_to_ids(sentence), train_labels[i])
                     for i, sentence in enumerate(clean_train_tokens)]
    test_reviews = [vocab.convert_tokens_to_ids(sentence)
                    for sentence in clean_test_tokens]

    train_reviews, val_reviews, train_labels, val_labels = train_test_split(
        train_reviews, train_labels, test_size=0.2, random_state=42)

    net = Transformer(
        vocab_size=len(vocab),
        embedding_dim=embed_size,
        hidden_dim=num_hiddens,
        num_class=labels,
        dim_feedforward=dim_feedforward,
        num_head=num_head,
        num_layers=num_layers,
        dropout=dropout_rate,
        max_len=512,
        activation="relu"
    )
    net.to(device)

    loss_function = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(net.parameters(), lr=lr, weight_decay=weight_decay)

    train_set = TransformerDataset(train_reviews)
    val_set = TransformerDataset(val_reviews)
    test_set = TransformerDataset(test_reviews)

    train_iter = torch.utils.data.DataLoader(train_set, collate_fn=collate_fn, batch_size=batch_size, shuffle=True, num_workers=0)
    val_iter = torch.utils.data.DataLoader(val_set, collate_fn=collate_fn, batch_size=batch_size, shuffle=False, num_workers=0)
    test_iter = torch.utils.data.DataLoader(test_set, collate_fn=collate_fn_test, batch_size=batch_size, shuffle=False, num_workers=0)

    global_step = 0
    best_val_acc = 0.0
    best_model_path = os.path.join(OUTPUT_DIR,"best_transformer.pt")

    for epoch in range(num_epochs):
        start = time.time()
        train_loss, val_losses = 0.0, 0.0
        train_acc, val_acc = 0.0, 0.0
        n, m = 0, 0

        net.train()
        with tqdm(total=len(train_iter), desc=f'Epoch {epoch}') as pbar:
            for feature, lengths, label in train_iter:
                n += 1
                global_step +=1
                # warmup lr
                lr_now = get_lr(global_step, warmup_steps, num_hiddens)
                for param_group in optimizer.param_groups:
                    param_group['lr'] = lr_now

                optimizer.zero_grad()
                feature = feature.to(device)
                lengths = lengths.to(device)
                label = label.to(device)
                score = net(feature, lengths)
                loss = loss_function(score, label)
                loss.backward()
                nn.utils.clip_grad_norm_(net.parameters(), grad_clip)
                optimizer.step()

                train_acc += accuracy_score(torch.argmax(score.cpu().data, dim=1), label.cpu())
                train_loss += loss.item()

                pbar.set_postfix({
                    'loss': f'{train_loss / n:.4f}',
                    'acc': f'{train_acc / n:.3f}',
                    'lr':f'{lr_now:.6f}'
                })
                pbar.update(1)

        net.eval()
        with torch.no_grad():
            for val_feature, val_length, val_label in val_iter:
                m += 1
                val_feature = val_feature.to(device)
                val_length = val_length.to(device)
                val_label = val_label.to(device)
                val_score = net(val_feature, val_length)
                val_loss = loss_function(val_score, val_label)
                val_acc += accuracy_score(torch.argmax(val_score.cpu().data, dim=1), val_label.cpu())
                val_losses += val_loss.item()

        train_avg_loss = train_loss / n
        train_avg_acc = train_acc / n
        val_avg_loss = val_losses / m
        val_avg_acc = val_acc / m
        end = time.time()
        runtime = end - start
        print(f"Epoch {epoch} | train_loss:{train_avg_loss:.4f} train_acc:{train_avg_acc:.4f} | val_loss:{val_avg_loss:.4f} val_acc:{val_avg_acc:.4f} | time:{runtime:.2f}s")

        # 早停保存最优模型
        if val_avg_acc > best_val_acc:
            best_val_acc = val_avg_acc
            torch.save(net.state_dict(), best_model_path)
            logger.info(f"save best model, best val acc {best_val_acc:.4f}")

    # 加载最优权重做预测
    net.load_state_dict(torch.load(best_model_path, map_location=device))
    net.eval()
    test_pred = []
    with torch.no_grad():
        with tqdm(total=len(test_iter), desc='Prediction') as pbar:
            for test_feature, test_len in test_iter:
                test_feature = test_feature.to(device)
                test_len = test_len.to(device)
                test_score = net(test_feature, test_len)
                test_pred.extend(torch.argmax(test_score.cpu().data, dim=1).numpy().tolist())
                pbar.update(1)

    result_output = pd.DataFrame(data={"id": test["id"], "sentiment": test_pred})
    save_path = os.path.join(OUTPUT_DIR, "transformer_best.csv")
    result_output.to_csv(save_path, index=False, quoting=3)
    logger.info(f'result saved to {save_path}, best val acc {best_val_acc:.4f}')


2026-08-20 07:38:58,422: INFO: vocab size: 38839
/tmp/ipykernel_58/1528070865.py:136: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers)
Epoch 0: 100%|██████████| 625/625 [00:55<00:00, 11.24it/s, loss=0.5897, acc=0.679, lr=0.003536]
2026-08-20 07:40:00,104: INFO: save best model, best val acc 0.7054


Epoch 0 | train_loss:0.5897 train_acc:0.6790 | val_loss:0.5731 val_acc:0.7054 | time:59.28s


Epoch 1: 100%|██████████| 625/625 [00:54<00:00, 11.39it/s, loss=0.6444, acc=0.606, lr=0.002500]


Epoch 1 | train_loss:0.6444 train_acc:0.6061 | val_loss:0.6986 val_acc:0.4974 | time:58.50s


Epoch 2: 100%|██████████| 625/625 [00:55<00:00, 11.34it/s, loss=0.6940, acc=0.500, lr=0.002041]


...

Epoch 10: 100%|██████████| 625/625 [00:55<00:00, 11.33it/s, loss=0.6934, acc=0.496, lr=0.001066]


Epoch 10 | train_loss:0.6934 train_acc:0.4963 | val_loss:0.6933 val_acc:0.4974 | time:58.83s


Epoch 11: 100%|██████████| 625/625 [00:55<00:00, 11.32it/s, loss=0.6933, acc=0.498, lr=0.001021]


Epoch 11 | train_loss:0.6933 train_acc:0.4980 | val_loss:0.6932 val_acc:0.4974 | time:58.88s


Prediction: 100%|██████████| 782/782 [00:17<00:00, 43.79it/s]
2026-08-20 07:51:03,093: INFO: result saved to /kaggle/working/result/transformer_best.csv, best val acc 0.7054


In [8]:
import re
import pickle
import os
import logging
import time
from collections import Counter
from tqdm import tqdm
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score
from bs4 import BeautifulSoup

# ===================== 配置区 =====================
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# 只需要word2vec‑nlp‑tutorial数据集
TRAIN_PATH = "/kaggle/input/competitions/word2vec-nlp-tutorial/labeledTrainData.tsv.zip"
TEST_PATH = "/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip"

PICKLE_FILE = "/kaggle/working/imdb_transformer.pickle3"
MODEL_PATH = "/kaggle/working/best_transformer.pt"
SUBMIT_PATH = "/kaggle/working/submit.csv"

MAX_LEN = 250
VOCAB_MAX_SIZE = 20000   # 词典最大词数
EMB_DIM = 256
NUM_HEAD = 8
NUM_LAYERS = 4
FFN_DIM = 512
DROPOUT = 0.3

batch_size = 64
lr_base = 1e-4
weight_decay = 1e-4
num_epochs = 25
patience = 6
label_smoothing = 0.1

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 打印输入目录，排查文件
print("==== /kaggle/input 文件列表 ====")
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        print(os.path.join(root, f))
print("==================================")

# ===================== 文本预处理 =====================
def review_to_wordlist(review):
    text = BeautifulSoup(review, "html.parser").get_text()
    text = re.sub(r"[^a-zA-Z0-9']", " ", text)
    words = text.lower().split()
    return words

def tokens2ids(tokens_list, word2idx, max_len):
    out = []
    for tokens in tokens_list:
        ids = [word2idx.get(w, 1) for w in tokens[:max_len]]
        if len(ids) < max_len:
            ids += [0] * (max_len - len(ids))
        out.append(ids)
    return np.array(out, dtype=np.int64)

# ===================== Transformer模型：位置编码 + Encoder =====================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-np.log(10000.0) / d_model))
        pe = torch.zeros(1, max_len, d_model)
        pe[0, :, 0::2] = torch.sin(position * div_term)
        pe[0, :, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x: [B, L, D]
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


class IMDBTransformer(nn.Module):
    def __init__(self, vocab_size, emb_dim, nhead, num_layers, ffn_dim, dropout, max_len):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.pos_encoder = PositionalEncoding(emb_dim, max_len, dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=emb_dim,
            nhead=nhead,
            dim_feedforward=ffn_dim,
            dropout=dropout,
            batch_first=True,
            activation="gelu"
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.norm = nn.LayerNorm(emb_dim)
        self.drop_fc = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(emb_dim, 128),
            nn.GELU(),
            nn.LayerNorm(128),
            nn.Dropout(dropout),
            nn.Linear(128, 2)
        )

    def forward(self, input_ids):
        B, L = input_ids.shape
        pad_mask = (input_ids == 0)  # padding mask True表示pad位置

        x = self.embedding(input_ids)
        x = self.pos_encoder(x)
        x = self.transformer_encoder(x, src_key_padding_mask=pad_mask)

        # 取[CLS]风格：对序列做平均池化
        mask_non_pad = (~pad_mask).unsqueeze(-1).float()
        sum_feat = torch.sum(x * mask_non_pad, dim=1)
        len_valid = torch.clamp(mask_non_pad.sum(dim=1), min=1e-6)
        pool = sum_feat / len_valid

        pool = self.norm(pool)
        pool = self.drop_fc(pool)
        logits = self.classifier(pool)
        return logits

# ===================== 主流程 =====================
if __name__ == "__main__":
    logging.basicConfig(format="%(asctime)s %(levelname)s: %(message)s", level=logging.INFO)
    logger = logging.getLogger(__name__)

    logger.info("Reading raw data...")
    train_df = pd.read_csv(TRAIN_PATH, sep="\t")
    test_df = pd.read_csv(TEST_PATH, sep="\t")

    train_tokens = [review_to_wordlist(r) for r in train_df["review"]]
    test_tokens = [review_to_wordlist(r) for r in test_df["review"]]

    # 构建词典
    all_words = []
    for sent in train_tokens:
        all_words.extend(sent)
    counter = Counter(all_words)
    # 保留高频词
    vocab_list = [w for w, _ in counter.most_common(VOCAB_MAX_SIZE - 2)]
    word2idx = {"<PAD>": 0, "<UNK>": 1}
    for w in vocab_list:
        word2idx[w] = len(word2idx)
    vocab_size = len(word2idx)
    logger.info(f"Vocab size: {vocab_size}")

    train_x = tokens2ids(train_tokens, word2idx, MAX_LEN)
    train_y = train_df["sentiment"].values.astype(np.int64)
    test_x = tokens2ids(test_tokens, word2idx, MAX_LEN)

    # 划分训练/验证
    split = int(0.8 * len(train_x))
    val_x, val_y = train_x[split:], train_y[split:]
    train_x, train_y = train_x[:split], train_y[:split]
    logger.info(f"train:{train_x.shape}, val:{val_x.shape}, test:{test_x.shape}")

    # 保存pickle
    save_data = {
        "train_x": train_x, "train_y": train_y,
        "val_x": val_x, "val_y": val_y,
        "test_x": test_x, "test_df": test_df,
        "word2idx": word2idx
    }
    with open(PICKLE_FILE, "wb") as f:
        pickle.dump(save_data, f)
    logger.info(f"Pickle saved {PICKLE_FILE}")

    # 加载数据集
    with open(PICKLE_FILE, "rb") as f:
        data = pickle.load(f)
    train_x, train_y = data["train_x"], data["train_y"]
    val_x, val_y = data["val_x"], data["val_y"]
    test_x = data["test_x"]
    test_df = data["test_df"]
    vocab_size = len(data["word2idx"])

    train_ds = torch.utils.data.TensorDataset(torch.from_numpy(train_x), torch.from_numpy(train_y))
    val_ds = torch.utils.data.TensorDataset(torch.from_numpy(val_x), torch.from_numpy(val_y))
    test_ds = torch.utils.data.TensorDataset(torch.from_numpy(test_x))

    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
    val_loader = torch.utils.data.DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0)
    test_loader = torch.utils.data.DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=0)

    model = IMDBTransformer(
        vocab_size=vocab_size,
        emb_dim=EMB_DIM,
        nhead=NUM_HEAD,
        num_layers=NUM_LAYERS,
        ffn_dim=FFN_DIM,
        dropout=DROPOUT,
        max_len=MAX_LEN
    ).to(device)

    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    optimizer = optim.AdamW(model.parameters(), lr=lr_base, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=2, eta_min=1e-5)

    best_acc = 0.0
    best_epoch = 0
    stop_cnt = 0

    for epoch in range(num_epochs):
        t0 = time.time()
        model.train()
        total_tr_loss, total_tr_acc, cnt = 0.0, 0.0, 0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch}")
        for bx, by in pbar:
            cnt += 1
            bx, by = bx.to(device), by.to(device)
            optimizer.zero_grad()
            logits = model(bx)
            loss = criterion(logits, by)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()

            pred = torch.argmax(logits.cpu(), dim=-1)
            total_tr_acc += accuracy_score(pred, by.cpu())
            total_tr_loss += loss.item()
            pbar.set_postfix({"loss": f"{total_tr_loss/cnt:.4f}", "acc": f"{total_tr_acc/cnt:.4f}"})
        scheduler.step()

        # 验证
        model.eval()
        total_val_loss, total_val_acc, m = 0.0, 0.0, 0
        with torch.no_grad():
            for bx, by in val_loader:
                m += 1
                bx, by = bx.to(device), by.to(device)
                logits = model(bx)
                lv = criterion(logits, by)
                total_val_loss += lv.item()
                pv = torch.argmax(logits.cpu(), dim=-1)
                total_val_acc += accuracy_score(pv, by.cpu())

        tr_loss_avg, tr_acc_avg = total_tr_loss / cnt, total_tr_acc / cnt
        va_loss_avg, va_acc_avg = total_val_loss / m, total_val_acc / m
        lr_now = optimizer.param_groups[0]["lr"]
        print(f"==== Epoch {epoch} lr={lr_now:.6f} time={time.time()-t0:.2f}s ====")
        print(f"Train loss:{tr_loss_avg:.4f} acc:{tr_acc_avg:.4f} | Val loss:{va_loss_avg:.4f} acc:{va_acc_avg:.4f}")

        if va_acc_avg > best_acc:
            best_acc = va_acc_avg
            best_epoch = epoch
            stop_cnt = 0
            torch.save(model.state_dict(), MODEL_PATH)
            print(f"★ New best val acc {best_acc:.4f} saved")
        else:
            stop_cnt += 1
            if stop_cnt >= patience:
                print(f"Early stop. Best acc={best_acc:.4f} @epoch {best_epoch}")
                break

    # 测试集预测输出提交文件
    logger.info(f"Load best model predict test, best val acc {best_acc:.4f}")
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    model.eval()
    test_preds = []
    with torch.no_grad():
        for bx, in tqdm(test_loader, desc="Predicting"):
            bx = bx.to(device)
            logits = model(bx)
            pred = torch.argmax(logits.cpu(), dim=-1).numpy().tolist()
            test_preds.extend(pred)

    out_df = pd.DataFrame({"id": test_df["id"], "sentiment": test_preds})
    out_df.to_csv(SUBMIT_PATH, index=False)
    logger.info(f"Submit file saved: {SUBMIT_PATH}")


INFO:__main__:Reading raw data...


==== /kaggle/input 文件列表 ====
/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip
/kaggle/input/competitions/word2vec-nlp-tutorial/sampleSubmission.csv
/kaggle/input/competitions/word2vec-nlp-tutorial/unlabeledTrainData.tsv.zip
/kaggle/input/competitions/word2vec-nlp-tutorial/labeledTrainData.tsv.zip


INFO:__main__:Vocab size: 20000
INFO:__main__:train:(20000, 250), val:(5000, 250), test:(25000, 250)
INFO:__main__:Pickle saved /kaggle/working/imdb_transformer.pickle3
Epoch 0: 100%|██████████| 313/313 [00:46<00:00,  6.74it/s, loss=0.6762, acc=0.5765]
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


==== Epoch 0 lr=0.000091 time=48.56s ====
Train loss:0.6762 acc:0.5765 | Val loss:0.5963 acc:0.6986
★ New best val acc 0.6986 saved


Epoch 1: 100%|██████████| 313/313 [00:49<00:00,  6.36it/s, loss=0.5633, acc=0.7359]


...

Epoch 16: 100%|██████████| 313/313 [00:49<00:00,  6.38it/s, loss=0.4110, acc=0.8639]


==== Epoch 16 lr=0.000098 time=51.14s ====
Train loss:0.4110 acc:0.8639 | Val loss:0.4451 acc:0.8406


Epoch 17: 100%|██████████| 313/313 [00:49<00:00,  6.39it/s, loss=0.4056, acc=0.8675]
INFO:__main__:Load best model predict test, best val acc 0.8485


==== Epoch 17 lr=0.000095 time=51.12s ====
Train loss:0.4056 acc:0.8675 | Val loss:0.4339 acc:0.8467
Early stop. Best acc=0.8485 @epoch 11


Predicting: 100%|██████████| 391/391 [00:10<00:00, 37.71it/s]
INFO:__main__:Submit file saved: /kaggle/working/submit.csv


In [9]:
import re
import pickle
import os
import logging
import time
from collections import Counter
from tqdm import tqdm
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score
from bs4 import BeautifulSoup

# ===================== 配置区 =====================
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Kaggle比赛数据集，zip压缩包，开启compression
TRAIN_PATH = "/kaggle/input/competitions/word2vec-nlp-tutorial/labeledTrainData.tsv.zip"
TEST_PATH = "/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip"

PICKLE_FILE = "/kaggle/working/imdb_transformer.pickle3"
MODEL_PATH = "/kaggle/working/best_transformer.pt"
CKPT_PATH = "/kaggle/working/latest_ckpt.pt"
SUBMIT_PATH = "/kaggle/working/submit.csv"

MAX_LEN = 250
VOCAB_MAX_SIZE = 20000
EMB_DIM = 256
NUM_HEAD = 8
NUM_LAYERS = 4
FFN_DIM = 512
DROPOUT = 0.3

batch_size = 64
lr_base = 1e-4
warmup_steps = 300
weight_decay = 1e-4
num_epochs = 25
patience = 7
label_smoothing = 0.1

# 特殊token id
PAD_ID = 0
UNK_ID = 1
CLS_ID = 2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 打印输入目录，排查文件
print("==== /kaggle/input 文件列表 ====")
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        print(os.path.join(root, f))
print("==================================")

# ===================== 文本预处理 =====================
def review_to_wordlist(review):
    text = BeautifulSoup(review, "html.parser").get_text()
    text = re.sub(r"[^a-zA-Z0-9']", " ", text)
    words = text.lower().split()
    return words

def tokens2ids_cls(tokens_list, word2idx, max_len):
    """开头插入CLS token"""
    out = []
    for tokens in tokens_list:
        ids = [CLS_ID] + [word2idx.get(w, UNK_ID) for w in tokens[:max_len-1]]
        if len(ids) < max_len:
            ids += [PAD_ID] * (max_len - len(ids))
        out.append(ids)
    return np.array(out, dtype=np.int64)

# ===================== Transformer模型：位置编码 + Encoder Pre‑LN + CLS输出 =====================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-np.log(10000.0) / d_model))
        pe = torch.zeros(1, max_len, d_model)
        pe[0, :, 0::2] = torch.sin(position * div_term)
        pe[0, :, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x: [B, L, D]
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


class IMDBTransformer(nn.Module):
    def __init__(self, vocab_size, emb_dim, nhead, num_layers, ffn_dim, dropout, max_len):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_ID)
        self.pos_encoder = PositionalEncoding(emb_dim, max_len, dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=emb_dim,
            nhead=nhead,
            dim_feedforward=ffn_dim,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
            norm_first=True  # Pre‑LN，训练更稳定
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.norm = nn.LayerNorm(emb_dim)
        self.drop_fc = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(emb_dim, 128),
            nn.GELU(),
            nn.LayerNorm(128),
            nn.Dropout(dropout),
            nn.Linear(128, 2)
        )

    def forward(self, input_ids):
        B, L = input_ids.shape
        pad_mask = (input_ids == PAD_ID)

        x = self.embedding(input_ids)
        x = self.pos_encoder(x)
        x = self.transformer_encoder(x, src_key_padding_mask=pad_mask)

        # 取CLS位置第0位做分类
        cls_feat = x[:, 0, :]
        cls_feat = self.norm(cls_feat)
        cls_feat = self.drop_fc(cls_feat)
        logits = self.classifier(cls_feat)
        return logits


# 自定义warmup + cosine学习率调度器
class WarmupCosineLR:
    def __init__(self, optimizer, warmup, total_steps, lr_min):
        self.opt = optimizer
        self.warmup = warmup
        self.total = total_steps
        self.lr_min = lr_min
        self.base_lr = optimizer.param_groups[0]["lr"]
        self.step_cnt = 0

    def step(self):
        self.step_cnt += 1
        if self.step_cnt <= self.warmup:
            lr = self.base_lr * (self.step_cnt / self.warmup)
        else:
            progress = (self.step_cnt - self.warmup) / (self.total - self.warmup)
            lr = self.lr_min + 0.5 * (self.base_lr - self.lr_min) * (1 + np.cos(np.pi * progress))
        for g in self.opt.param_groups:
            g["lr"] = lr
        return lr

# ===================== 主流程 =====================
if __name__ == "__main__":
    logging.basicConfig(format="%(asctime)s %(levelname)s: %(message)s", level=logging.INFO)
    logger = logging.getLogger(__name__)

    logger.info("Reading raw data...")
    train_df = pd.read_csv(TRAIN_PATH, sep="\t", compression="zip")
    test_df = pd.read_csv(TEST_PATH, sep="\t", compression="zip")

    train_tokens = [review_to_wordlist(r) for r in train_df["review"]]
    test_tokens = [review_to_wordlist(r) for r in test_df["review"]]

    # 构建词典，预留CLS
    all_words = []
    for sent in train_tokens:
        all_words.extend(sent)
    counter = Counter(all_words)
    vocab_list = [w for w, _ in counter.most_common(VOCAB_MAX_SIZE - 3)]
    word2idx = {"<PAD>": PAD_ID, "<UNK>": UNK_ID, "<CLS>": CLS_ID}
    for w in vocab_list:
        word2idx[w] = len(word2idx)
    vocab_size = len(word2idx)
    logger.info(f"Vocab size: {vocab_size}")

    train_x = tokens2ids_cls(train_tokens, word2idx, MAX_LEN)
    train_y = train_df["sentiment"].values.astype(np.int64)
    test_x = tokens2ids_cls(test_tokens, word2idx, MAX_LEN)

    split = int(0.8 * len(train_x))
    val_x, val_y = train_x[split:], train_y[split:]
    train_x, train_y = train_x[:split], train_y[:split]
    logger.info(f"train:{train_x.shape}, val:{val_x.shape}, test:{test_x.shape}")

    save_data = {
        "train_x": train_x, "train_y": train_y,
        "val_x": val_x, "val_y": val_y,
        "test_x": test_x, "test_df": test_df,
        "word2idx": word2idx
    }
    with open(PICKLE_FILE, "wb") as f:
        pickle.dump(save_data, f)
    logger.info(f"Pickle saved {PICKLE_FILE}")

    with open(PICKLE_FILE, "rb") as f:
        data = pickle.load(f)
    train_x, train_y = data["train_x"], data["train_y"]
    val_x, val_y = data["val_x"], data["val_y"]
    test_x = data["test_x"]
    test_df = data["test_df"]
    vocab_size = len(data["word2idx"])

    train_ds = torch.utils.data.TensorDataset(torch.from_numpy(train_x), torch.from_numpy(train_y))
    val_ds = torch.utils.data.TensorDataset(torch.from_numpy(val_x), torch.from_numpy(val_y))
    test_ds = torch.utils.data.TensorDataset(torch.from_numpy(test_x))

    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
    val_loader = torch.utils.data.DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0)
    test_loader = torch.utils.data.DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=0)

    model = IMDBTransformer(
        vocab_size=vocab_size,
        emb_dim=EMB_DIM,
        nhead=NUM_HEAD,
        num_layers=NUM_LAYERS,
        ffn_dim=FFN_DIM,
        dropout=DROPOUT,
        max_len=MAX_LEN
    ).to(device)

    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    optimizer = optim.AdamW(model.parameters(), lr=lr_base, weight_decay=weight_decay)

    total_train_steps = num_epochs * len(train_loader)
    scheduler = WarmupCosineLR(optimizer, warmup=warmup_steps, total_steps=total_train_steps, lr_min=1e-5)

    best_acc = 0.0
    best_epoch = 0
    stop_cnt = 0

    for epoch in range(num_epochs):
        t0 = time.time()
        model.train()
        total_tr_loss, total_tr_acc, cnt = 0.0, 0.0, 0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch}")
        for bx, by in pbar:
            cnt += 1
            bx, by = bx.to(device), by.to(device)
            optimizer.zero_grad()
            logits = model(bx)
            loss = criterion(logits, by)

            # 检测NaN loss
            if torch.isnan(loss):
                logger.warning("NaN loss detected, skip batch")
                continue

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            scheduler.step()

            pred = torch.argmax(logits.cpu(), dim=-1)
            total_tr_acc += accuracy_score(pred, by.cpu())
            total_tr_loss += loss.item()
            current_lr = optimizer.param_groups[0]["lr"]
            pbar.set_postfix({"loss": f"{total_tr_loss/cnt:.4f}", "acc": f"{total_tr_acc/cnt:.4f}", "lr": f"{current_lr:.6f}"})

        # 保存完整checkpoint
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "opt_state_dict": optimizer.state_dict(),
            "best_acc": best_acc
        }, CKPT_PATH)

        model.eval()
        total_val_loss, total_val_acc, m = 0.0, 0.0, 0
        with torch.no_grad():
            for bx, by in val_loader:
                m += 1
                bx, by = bx.to(device), by.to(device)
                logits = model(bx)
                lv = criterion(logits, by)
                total_val_loss += lv.item()
                pv = torch.argmax(logits.cpu(), dim=-1)
                total_val_acc += accuracy_score(pv, by.cpu())

        tr_loss_avg, tr_acc_avg = total_tr_loss / cnt, total_tr_acc / cnt
        va_loss_avg, va_acc_avg = total_val_loss / m, total_val_acc / m
        lr_now = optimizer.param_groups[0]["lr"]
        print(f"==== Epoch {epoch} lr={lr_now:.6f} time={time.time()-t0:.2f}s ====")
        print(f"Train loss:{tr_loss_avg:.4f} acc:{tr_acc_avg:.4f} | Val loss:{va_loss_avg:.4f} acc:{va_acc_avg:.4f}")

        if va_acc_avg > best_acc:
            best_acc = va_acc_avg
            best_epoch = epoch
            stop_cnt = 0
            torch.save(model.state_dict(), MODEL_PATH)
            print(f"★ New best val acc {best_acc:.4f} saved")
        else:
            stop_cnt += 1
            if stop_cnt >= patience:
                print(f"Early stop. Best acc={best_acc:.4f} @epoch {best_epoch}")
                break

    logger.info(f"Load best model predict test, best val acc {best_acc:.4f}")
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    model.eval()
    test_preds = []
    with torch.no_grad():
        for bx, in tqdm(test_loader, desc="Predicting"):
            bx = bx.to(device)
            logits = model(bx)
            pred = torch.argmax(logits.cpu(), dim=-1).numpy().tolist()
            test_preds.extend(pred)

    out_df = pd.DataFrame({"id": test_df["id"], "sentiment": test_preds})
    out_df.to_csv(SUBMIT_PATH, index=False)
    logger.info(f"Submit file saved: {SUBMIT_PATH}")


INFO:__main__:Reading raw data...


==== /kaggle/input 文件列表 ====
/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip
/kaggle/input/competitions/word2vec-nlp-tutorial/sampleSubmission.csv
/kaggle/input/competitions/word2vec-nlp-tutorial/unlabeledTrainData.tsv.zip
/kaggle/input/competitions/word2vec-nlp-tutorial/labeledTrainData.tsv.zip


INFO:__main__:Vocab size: 20000
INFO:__main__:train:(20000, 250), val:(5000, 250), test:(25000, 250)
INFO:__main__:Pickle saved /kaggle/working/imdb_transformer.pickle3
/tmp/ipykernel_58/3496940663.py:113: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
Epoch 0: 100%|██████████| 313/313 [00:49<00:00,  6.30it/s, loss=0.7408, acc=0.5042, lr=0.000100]


==== Epoch 0 lr=0.000100 time=53.37s ====
Train loss:0.7408 acc:0.5042 | Val loss:0.6904 acc:0.5077
★ New best val acc 0.5077 saved


Epoch 1: 100%|██████████| 313/313 [00:48<00:00,  6.46it/s, loss=0.6966, acc=0.5237, lr=0.000100]


...

Epoch 21: 100%|██████████| 313/313 [00:48<00:00,  6.42it/s, loss=0.3913, acc=0.8796, lr=0.000013]


==== Epoch 21 lr=0.000013 time=52.66s ====
Train loss:0.3913 acc:0.8796 | Val loss:0.4609 acc:0.8542


Epoch 22: 100%|██████████| 313/313 [00:48<00:00,  6.43it/s, loss=0.3909, acc=0.8796, lr=0.000012]
INFO:__main__:Load best model predict test, best val acc 0.8562


==== Epoch 22 lr=0.000012 time=52.55s ====
Train loss:0.3909 acc:0.8796 | Val loss:0.4579 acc:0.8536
Early stop. Best acc=0.8562 @epoch 15


Predicting: 100%|██████████| 391/391 [00:18<00:00, 21.30it/s]
INFO:__main__:Submit file saved: /kaggle/working/submit.csv


In [2]:
import re
import pickle
import os
import logging
import time
from collections import Counter
from tqdm import tqdm
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score, f1_score
from bs4 import BeautifulSoup

# ===================== 配置区 =====================
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

TRAIN_PATH = "/kaggle/input/competitions/word2vec-nlp-tutorial/labeledTrainData.tsv.zip"
TEST_PATH = "/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip"

PICKLE_FILE = "/kaggle/working/imdb_transformer.pickle3"
MODEL_PATH = "/kaggle/working/best_transformer.pt"
CKPT_PATH = "/kaggle/working/latest_ckpt.pt"
SUBMIT_PATH = "/kaggle/working/submit.csv"

MAX_LEN = 250
VOCAB_MAX_SIZE = 20000
EMB_DIM = 256
NUM_HEAD = 8
NUM_LAYERS = 4
FFN_DIM = 512
DROPOUT = 0.35
EMB_DROP = 0.2

batch_size = 64
lr_base = 8e-5
warmup_steps = 350
weight_decay = 1.2e-4
num_epochs = 28
patience = 8
label_smoothing = 0.1
word_dropout_rate = 0.15  # 训练时随机替换部分词为UNK，数据增强

PAD_ID = 0
UNK_ID = 1
CLS_ID = 2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("==== /kaggle/input 文件列表 ====")
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        print(os.path.join(root, f))
print("==================================")

# ===================== 文本预处理 =====================
def review_to_wordlist(review):
    text = BeautifulSoup(review, "html.parser").get_text()
    text = re.sub(r"[^a-zA-Z0-9']", " ", text)
    words = text.lower().split()
    return words

def tokens2ids_cls(tokens_list, word2idx, max_len):
    out = []
    for tokens in tokens_list:
        ids = [CLS_ID] + [word2idx.get(w, UNK_ID) for w in tokens[:max_len-1]]
        if len(ids) < max_len:
            ids += [PAD_ID] * (max_len - len(ids))
        out.append(ids)
    return np.array(out, dtype=np.int64)

def word_dropout(x_np, rate, unk_id):
    """训练集numpy数组，随机把非PAD、非CLS替换成UNK，数据增强"""
    mask = np.random.random(x_np.shape) < rate
    pad_mask = (x_np == PAD_ID)
    cls_mask = (x_np == CLS_ID)
    mask = mask & (~pad_mask) & (~cls_mask)
    x_noise = x_np.copy()
    x_noise[mask] = unk_id
    return x_noise

# ===================== Transformer模型：Pre‑LN + CLS + EmbeddingDropout =====================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-np.log(10000.0) / d_model))
        pe = torch.zeros(1, max_len, d_model)
        pe[0, :, 0::2] = torch.sin(position * div_term)
        pe[0, :, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


class IMDBTransformer(nn.Module):
    def __init__(self, vocab_size, emb_dim, nhead, num_layers, ffn_dim, dropout, emb_drop, max_len):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_ID)
        self.emb_dropout = nn.Dropout(emb_drop)
        self.pos_encoder = PositionalEncoding(emb_dim, max_len, dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=emb_dim,
            nhead=nhead,
            dim_feedforward=ffn_dim,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
            norm_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.norm = nn.LayerNorm(emb_dim)
        self.drop_fc = nn.Dropout(dropout)
        self.proj = nn.Linear(emb_dim, 128)
        self.act = nn.GELU()
        self.ln2 = nn.LayerNorm(128)
        self.drop2 = nn.Dropout(dropout)
        self.classifier = nn.Linear(128, 2)

        # 权重初始化
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, input_ids):
        B, L = input_ids.shape
        pad_mask = (input_ids == PAD_ID)

        x = self.embedding(input_ids)
        x = self.emb_dropout(x)
        x = self.pos_encoder(x)
        x = self.transformer_encoder(x, src_key_padding_mask=pad_mask)

        cls_feat = x[:, 0, :]
        cls_feat = self.norm(cls_feat)

        h = self.proj(cls_feat)
        h = self.act(h)
        h = self.ln2(h)
        h = self.drop2(h)
        logits = self.classifier(h)
        return logits


# Warmup‑Cosine 学习率
class WarmupCosineLR:
    def __init__(self, optimizer, warmup, total_steps, lr_min):
        self.opt = optimizer
        self.warmup = warmup
        self.total = total_steps
        self.lr_min = lr_min
        self.base_lr = optimizer.param_groups[0]["lr"]
        self.step_cnt = 0

    def step(self):
        self.step_cnt += 1
        if self.step_cnt <= self.warmup:
            lr = self.base_lr * (self.step_cnt / self.warmup)
        else:
            progress = (self.step_cnt - self.warmup) / max(self.total - self.warmup, 1)
            lr = self.lr_min + 0.5 * (self.base_lr - self.lr_min) * (1 + np.cos(np.pi * progress))
        for g in self.opt.param_groups:
            g["lr"] = lr
        return lr

# ===================== 主流程 =====================
if __name__ == "__main__":
    logging.basicConfig(format="%(asctime)s %(levelname)s: %(message)s", level=logging.INFO)
    logger = logging.getLogger(__name__)

    logger.info("Reading raw data...")
    train_df = pd.read_csv(TRAIN_PATH, sep="\t", compression="zip")
    test_df = pd.read_csv(TEST_PATH, sep="\t", compression="zip")

    train_tokens = [review_to_wordlist(r) for r in train_df["review"]]
    test_tokens = [review_to_wordlist(r) for r in test_df["review"]]

    all_words = []
    for sent in train_tokens:
        all_words.extend(sent)
    counter = Counter(all_words)
    vocab_list = [w for w, _ in counter.most_common(VOCAB_MAX_SIZE - 3)]
    word2idx = {"<PAD>": PAD_ID, "<UNK>": UNK_ID, "<CLS>": CLS_ID}
    for w in vocab_list:
        word2idx[w] = len(word2idx)
    vocab_size = len(word2idx)
    logger.info(f"Vocab size: {vocab_size}")

    train_x = tokens2ids_cls(train_tokens, word2idx, MAX_LEN)
    train_y = train_df["sentiment"].values.astype(np.int64)
    test_x = tokens2ids_cls(test_tokens, word2idx, MAX_LEN)

    # 分层随机划分，保证正负样本比例
    np.random.shuffle(train_x)
    perm = np.random.permutation(len(train_x))
    split = int(0.8 * len(train_x))
    train_x, val_x = train_x[perm[:split]], train_x[perm[split:]]
    train_y, val_y = train_y[perm[:split]], train_y[perm[split:]]
    logger.info(f"train:{train_x.shape}, val:{val_x.shape}, test:{test_x.shape}")

    save_data = {
        "train_x": train_x, "train_y": train_y,
        "val_x": val_x, "val_y": val_y,
        "test_x": test_x, "test_df": test_df,
        "word2idx": word2idx
    }
    with open(PICKLE_FILE, "wb") as f:
        pickle.dump(save_data, f)

    with open(PICKLE_FILE, "rb") as f:
        data = pickle.load(f)
    train_x, train_y = data["train_x"], data["train_y"]
    val_x, val_y = data["val_x"], data["val_y"]
    test_x = data["test_x"]
    test_df = data["test_df"]
    vocab_size = len(data["word2idx"])

    train_ds = torch.utils.data.TensorDataset(torch.from_numpy(train_x), torch.from_numpy(train_y))
    val_ds = torch.utils.data.TensorDataset(torch.from_numpy(val_x), torch.from_numpy(val_y))
    test_ds = torch.utils.data.TensorDataset(torch.from_numpy(test_x))

    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
    val_loader = torch.utils.data.DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0)
    test_loader = torch.utils.data.DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=0)

    model = IMDBTransformer(
        vocab_size=vocab_size,
        emb_dim=EMB_DIM,
        nhead=NUM_HEAD,
        num_layers=NUM_LAYERS,
        ffn_dim=FFN_DIM,
        dropout=DROPOUT,
        emb_drop=EMB_DROP,
        max_len=MAX_LEN
    ).to(device)

    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    optimizer = optim.AdamW(model.parameters(), lr=lr_base, weight_decay=weight_decay)

    total_train_steps = num_epochs * len(train_loader)
    scheduler = WarmupCosineLR(optimizer, warmup=warmup_steps, total_steps=total_train_steps, lr_min=1e-5)

    best_acc = 0.0
    best_f1 = 0.0
    best_epoch = 0
    stop_cnt = 0

    for epoch in range(num_epochs):
        t0 = time.time()
        model.train()
        total_tr_loss = 0.0
        all_tr_pred = []
        all_tr_true = []
        cnt = 0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch}")
        for bx, by in pbar:
            cnt += 1
            # 每轮做word dropout增强
            bx_np = bx.numpy()
            bx_noise = word_dropout(bx_np, word_dropout_rate, UNK_ID)
            bx = torch.from_numpy(bx_noise).to(device)
            by = by.to(device)

            optimizer.zero_grad()
            logits = model(bx)
            loss = criterion(logits, by)

            if torch.isnan(loss):
                logger.warning("NaN loss detected, skip batch")
                continue

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            scheduler.step()

            pred = torch.argmax(logits, dim=-1)
            all_tr_pred.extend(pred.cpu().numpy().tolist())
            all_tr_true.extend(by.cpu().numpy().tolist())
            total_tr_loss += loss.item()
            current_lr = optimizer.param_groups[0]["lr"]
            pbar.set_postfix({"loss": f"{total_tr_loss/cnt:.4f}", "lr": f"{current_lr:.6f}"})

        tr_acc = accuracy_score(all_tr_true, all_tr_pred)
        tr_f1 = f1_score(all_tr_true, all_tr_pred)
        tr_loss_avg = total_tr_loss / cnt

        # 验证
        model.eval()
        total_val_loss = 0.0
        all_val_pred = []
        all_val_true = []
        m = 0
        with torch.no_grad():
            for bx, by in val_loader:
                m += 1
                bx, by = bx.to(device), by.to(device)
                logits = model(bx)
                lv = criterion(logits, by)
                total_val_loss += lv.item()
                pv = torch.argmax(logits, dim=-1)
                all_val_pred.extend(pv.cpu().numpy().tolist())
                all_val_true.extend(by.cpu().numpy().tolist())

        va_acc = accuracy_score(all_val_true, all_val_pred)
        va_f1 = f1_score(all_val_true, all_val_pred)
        va_loss_avg = total_val_loss / m
        lr_now = optimizer.param_groups[0]["lr"]

        print(f"==== Epoch {epoch} lr={lr_now:.6f} time={time.time()-t0:.2f}s ====")
        print(f"Train loss:{tr_loss_avg:.4f} acc:{tr_acc:.4f} f1:{tr_f1:.4f} | Val loss:{va_loss_avg:.4f} acc:{va_acc:.4f} f1:{va_f1:.4f}")

        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "opt_state_dict": optimizer.state_dict(),
            "best_acc": best_acc
        }, CKPT_PATH)

        if va_acc > best_acc:
            best_acc = va_acc
            best_f1 = va_f1
            best_epoch = epoch
            stop_cnt = 0
            torch.save(model.state_dict(), MODEL_PATH)
            print(f"★ New best val acc {best_acc:.4f} f1 {best_f1:.4f} saved")
        else:
            stop_cnt += 1
            if stop_cnt >= patience:
                print(f"Early stop. Best acc={best_acc:.4f}, best f1={best_f1:.4f} @epoch {best_epoch}")
                break

    logger.info(f"Load best model predict test, best val acc {best_acc:.4f}")
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    model.eval()
    test_preds = []
    with torch.no_grad():
        for bx, in tqdm(test_loader, desc="Predicting"):
            bx = bx.to(device)
            logits = model(bx)
            pred = torch.argmax(logits.cpu(), dim=-1).numpy().tolist()
            test_preds.extend(pred)

    out_df = pd.DataFrame({"id": test_df["id"], "sentiment": test_preds})
    out_df.to_csv(SUBMIT_PATH, index=False)
    logger.info(f"Submit file saved: {SUBMIT_PATH}")
    logger.info(f"==== Summary ====")
    logger.info(f"Best epoch:{best_epoch}, best val acc:{best_acc:.4f}, best val f1:{best_f1:.4f}")


2026-08-20 12:39:35,699 INFO: Reading raw data...


==== /kaggle/input 文件列表 ====
/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip
/kaggle/input/competitions/word2vec-nlp-tutorial/sampleSubmission.csv
/kaggle/input/competitions/word2vec-nlp-tutorial/unlabeledTrainData.tsv.zip
/kaggle/input/competitions/word2vec-nlp-tutorial/labeledTrainData.tsv.zip


2026-08-20 12:39:50,157 INFO: Vocab size: 20000
2026-08-20 12:39:52,635 INFO: train:(20000, 250), val:(5000, 250), test:(25000, 250)
/tmp/ipykernel_58/3032745751.py:121: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
Epoch 0: 100%|██████████| 313/313 [00:45<00:00,  6.86it/s, loss=0.9953, lr=0.000072]


==== Epoch 0 lr=0.000072 time=49.15s ====
Train loss:0.9953 acc:0.4978 f1:0.5022 | Val loss:0.6942 acc:0.4964 f1:0.6203
★ New best val acc 0.4964 f1 0.6203 saved


Epoch 1: 100%|██████████| 313/313 [00:47<00:00,  6.63it/s, loss=0.7469, lr=0.000080]


...

Epoch 9: 100%|██████████| 313/313 [00:46<00:00,  6.70it/s, loss=0.6966, lr=0.000063]


==== Epoch 9 lr=0.000063 time=50.25s ====
Train loss:0.6966 acc:0.5028 f1:0.5122 | Val loss:0.6957 acc:0.4848 f1:0.6530


2026-08-20 12:48:20,781 INFO: Load best model predict test, best val acc 0.5152


Early stop. Best acc=0.5152, best f1=0.0000 @epoch 1


Predicting: 100%|██████████| 391/391 [00:17<00:00, 22.12it/s]
2026-08-20 12:48:38,530 INFO: Submit file saved: /kaggle/working/submit.csv
2026-08-20 12:48:38,530 INFO: ==== Summary ====
2026-08-20 12:48:38,531 INFO: Best epoch:1, best val acc:0.5152, best val f1:0.0000


In [5]:
import re
import pickle
import os
import logging
import time
from collections import Counter
from tqdm import tqdm
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import autocast, GradScaler
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from bs4 import BeautifulSoup

# ===================== 配置 =====================
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

TRAIN_PATH = "/kaggle/input/competitions/word2vec-nlp-tutorial/labeledTrainData.tsv.zip"
TEST_PATH = "/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip"

PICKLE_FILE = "/kaggle/working/imdb.pkl"
MODEL_PATH = "/kaggle/working/best.pt"
SUBMIT_PATH = "/kaggle/working/submission.csv"

MAX_LEN = 250
VOCAB_MAX_SIZE = 20000
EMB_DIM = 256
NUM_HEAD = 8
NUM_LAYERS = 4
FFN_DIM = 512
DROPOUT = 0.3
EMB_DROP = 0.15

batch_size = 64
lr_base = 8e-5
warmup_steps = 300
weight_decay = 1e-4
num_epochs = 25
patience = 7
label_smoothing = 0.1
word_dropout_rate = 0.1

PAD_ID = 0
UNK_ID = 1
CLS_ID = 2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
scaler = GradScaler(enabled=torch.cuda.is_available())

logging.basicConfig(format="%(asctime)s %(levelname)s: %(message)s", level=logging.INFO)
logger = logging.getLogger(__name__)

# ===================== 预处理 =====================
def clean_review(review):
    text = BeautifulSoup(review, "html.parser").get_text()
    text = re.sub(r"[^a-zA-Z']", " ", text)
    return text.lower().split()

def tokens_to_ids(tokens_list, word2idx, max_len):
    out = []
    for tokens in tokens_list:
        ids = [CLS_ID] + [word2idx.get(w, UNK_ID) for w in tokens[:max_len-1]]
        if len(ids) < max_len:
            ids += [PAD_ID] * (max_len - len(ids))
        out.append(ids)
    return np.array(out, dtype=np.int64)

def add_word_noise(x_np, rate, unk_id):
    """训练时随机把部分非PAD/非CLS替换为UNK，简单数据增强"""
    mask = np.random.random(x_np.shape) < rate
    mask = mask & (x_np != PAD_ID) & (x_np != CLS_ID)
    x_noise = x_np.copy()
    x_noise[mask] = unk_id
    return x_noise

# ===================== 模型：可学习位置编码 Pre‑LN Transformer =====================
class LearnablePosEncoding(nn.Module):
    def __init__(self, max_len, d_model, dropout):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.pos_emb = nn.Embedding(max_len, d_model)
    def forward(self, x):
        B, L, D = x.shape
        pos = torch.arange(L, device=x.device).unsqueeze(0)
        x = x + self.pos_emb(pos)
        return self.dropout(x)

class IMDBTransformer(nn.Module):
    def __init__(self, vocab_size, emb_dim, nhead, num_layers, ffn_dim, dropout, emb_drop, max_len):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_ID)
        self.emb_drop = nn.Dropout(emb_drop)
        self.pos_enc = LearnablePosEncoding(max_len, emb_dim, dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=emb_dim,
            nhead=nhead,
            dim_feedforward=ffn_dim,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
            norm_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(emb_dim)
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(emb_dim, 2)
        )

        # 合理初始化
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, input_ids):
        pad_mask = (input_ids == PAD_ID)
        x = self.embedding(input_ids)
        x = self.emb_drop(x)
        x = self.pos_enc(x)
        x = self.encoder(x, src_key_padding_mask=pad_mask)
        cls_out = self.norm(x[:, 0, :])
        logits = self.classifier(cls_out)
        return logits

# ===================== 学习率调度 Warmup‑Cosine =====================
class WarmupCosineLR:
    def __init__(self, opt, warmup, total_steps, lr_min):
        self.opt = opt
        self.warmup = warmup
        self.total = total_steps
        self.lr_min = lr_min
        self.base_lr = opt.param_groups[0]["lr"]
        self.step_cnt = 0
    def step(self):
        self.step_cnt += 1
        if self.step_cnt <= self.warmup:
            lr = self.base_lr * (self.step_cnt / self.warmup)
        else:
            progress = (self.step_cnt - self.warmup) / max(self.total - self.warmup, 1)
            lr = self.lr_min + 0.5*(self.base_lr - self.lr_min)*(1+np.cos(np.pi*progress))
        for g in self.opt.param_groups:
            g["lr"] = lr
        return lr

# ===================== 主流程 =====================
if __name__ == "__main__":
    logger.info("load dataset")
    train_df = pd.read_csv(TRAIN_PATH, sep="\t", compression="zip")
    test_df = pd.read_csv(TEST_PATH, sep="\t", compression="zip")

    train_tokens = [clean_review(r) for r in train_df["review"]]
    test_tokens = [clean_review(r) for r in test_df["review"]]

    # 构建词表
    all_words = []
    for sent in train_tokens:
        all_words.extend(sent)
    counter = Counter(all_words)
    vocab_list = [w for w,_ in counter.most_common(VOCAB_MAX_SIZE-3)]
    word2idx = {"<PAD>":PAD_ID, "<UNK>":UNK_ID, "<CLS>":CLS_ID}
    for w in vocab_list:
        word2idx[w] = len(word2idx)
    vocab_size = len(word2idx)
    logger.info(f"vocab_size:{vocab_size}")

    train_x_all = tokens_to_ids(train_tokens, word2idx, MAX_LEN)
    train_y_all = train_df["sentiment"].values.astype(np.int64)
    test_x = tokens_to_ids(test_tokens, word2idx, MAX_LEN)

    # 分层划分，保证正负样本比例一致
    train_x, val_x, train_y, val_y = train_test_split(
        train_x_all, train_y_all, test_size=0.2, random_state=SEED, stratify=train_y_all
    )
    logger.info(f"train:{train_x.shape}, val:{val_x.shape}")

    ds_train = torch.utils.data.TensorDataset(torch.from_numpy(train_x), torch.from_numpy(train_y))
    ds_val = torch.utils.data.TensorDataset(torch.from_numpy(val_x), torch.from_numpy(val_y))
    ds_test = torch.utils.data.TensorDataset(torch.from_numpy(test_x))

    loader_train = torch.utils.data.DataLoader(ds_train, batch_size=batch_size, shuffle=True, num_workers=0)
    loader_val = torch.utils.data.DataLoader(ds_val, batch_size=batch_size, shuffle=False, num_workers=0)
    loader_test = torch.utils.data.DataLoader(ds_test, batch_size=batch_size, shuffle=False, num_workers=0)

    model = IMDBTransformer(vocab_size, EMB_DIM, NUM_HEAD, NUM_LAYERS, FFN_DIM, DROPOUT, EMB_DROP, MAX_LEN).to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    optimizer = optim.AdamW(model.parameters(), lr=lr_base, weight_decay=weight_decay)
    total_steps = num_epochs * len(loader_train)
    scheduler = WarmupCosineLR(optimizer, warmup_steps, total_steps, lr_min=1e-5)

    best_acc = 0.0
    best_f1 = 0.0
    best_epoch = 0
    early_stop_count = 0

    for epoch in range(num_epochs):
        t_start = time.time()
        model.train()
        loss_sum = 0.0
        pbar = tqdm(loader_train, desc=f"Epoch {epoch}")
        for bx, by in pbar:
            bx = torch.from_numpy(add_word_noise(bx.numpy(), word_dropout_rate, UNK_ID))
            bx, by = bx.to(device), by.to(device)
            optimizer.zero_grad()
            with autocast():
                logits = model(bx)
                loss = criterion(logits, by)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            loss_sum += loss.item()
            lr_now = optimizer.param_groups[0]["lr"]
            pbar.set_postfix({"loss":f"{loss_sum/(pbar.n+1):.4f}", "lr":f"{lr_now:.6f}"})

        # --------验证集评估--------
        model.eval()
        val_preds = []
        val_truth = []
        with torch.no_grad():
            for bx, by in loader_val:
                bx, by = bx.to(device), by.to(device)
                with autocast():
                    logits = model(bx)
                pred = torch.argmax(logits, dim=-1)
                val_preds.extend(pred.cpu().numpy())
                val_truth.extend(by.cpu().numpy())
        val_acc = accuracy_score(val_truth, val_preds)
        val_f1 = f1_score(val_truth, val_preds)
        cost = time.time()-t_start
        logger.info(f"Epoch {epoch} | val_acc:{val_acc:.4f} val_f1:{val_f1:.4f} time:{cost:.2f}s")

        if val_acc > best_acc:
            best_acc = val_acc
            best_f1 = val_f1
            best_epoch = epoch
            early_stop_count = 0
            torch.save(model.state_dict(), MODEL_PATH)
            logger.info(f"★ save best model acc={best_acc:.4f}")
        else:
            early_stop_count += 1
            if early_stop_count >= patience:
                logger.info(f"early-stop, best epoch={best_epoch}, best_acc={best_acc:.4f}, best_f1={best_f1:.4f}")
                break

    # 预测测试集输出提交文件
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    model.eval()
    test_out = []
    with torch.no_grad():
        for bx, in tqdm(loader_test, desc="predict test"):
            bx = bx.to(device)
            with autocast():
                logits = model(bx)
            pred = torch.argmax(logits, dim=-1).cpu().numpy()
            test_out.extend(pred)
    submit_df = pd.DataFrame({"id": test_df["id"], "sentiment": test_out})
    submit_df.to_csv(SUBMIT_PATH, index=False)
    logger.info(f"saved submission to {SUBMIT_PATH}")
    logger.info(f"Final summary: best_epoch={best_epoch}, best_val_acc={best_acc:.4f}, best_val_f1={best_f1:.4f}")


/tmp/ipykernel_58/2508982463.py:56: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=torch.cuda.is_available())
2026-08-20 12:54:08,048 INFO: load dataset
2026-08-20 12:54:21,800 INFO: vocab_size:20000
2026-08-20 12:54:23,772 INFO: train:(20000, 250), val:(5000, 250)
/tmp/ipykernel_58/2508982463.py:112: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
Epoch 0:   0%|          | 0/313 [00:00<?, ?it/s]/tmp/ipykernel_58/2508982463.py:214: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Epoch 0: 100%|██████████| 313/313 [00:20<00:00, 15.29it/s, loss=0.9350, lr=0.000080]
/tmp/ipykernel_58/2508982463.py:234: FutureWarning: `torch.cuda.amp.autocast(